

# Avance1.#Equipo  
## Avance 1. Análisis exploratorio de datos

**Nombre del proyecto:** Sports Bet Engine  
**Equipo:** 37  

**Integrantes:**  
- Jeanette Ríos Martínez — A01688888  
- Ali Mateo Campos Martínez — A01796071  
- Ali Alfonso Rico Vázquez — A01350630  

**Curso:** MNA - Maestría en Inteligencia Artificial Aplicada  
**Entregable:** Avance 1. Análisis exploratorio de datos  

---

## Objetivo de la libreta

El objetivo de este notebook es realizar un análisis exploratorio de datos para el proyecto **Sports Bet Engine**, integrando datos estructurados de marcadores deportivos y datos no estructurados provenientes de noticias deportivas.  

Este análisis permite evaluar la calidad de los datos, identificar patrones, revisar distribuciones y preparar una base inicial para modelos predictivos que puedan apoyar la toma de decisiones en mercados de pronósticos deportivos.


## 🔹 0. Instalación e importación de librerías

En esta sección se prepara el entorno de trabajo.  
Se instalan e importan las librerías necesarias para:

- Manipulación de datos.
- Visualización.
- Consulta de APIs.
- Procesamiento básico de texto.
- Análisis exploratorio.
- Modelado inicial.

También se define una semilla aleatoria para que los resultados simulados sean reproducibles.


In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn nltk requests nba_api --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import warnings

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams["figure.figsize"] = (9, 5)

print("Librerías cargadas correctamente.")


## 🔹 1. Fuentes de información del proyecto

De acuerdo con la propuesta del proyecto, se consideran dos tipos principales de datos:

### Datos estructurados
Son datos deportivos históricos, como puntos, marcadores, diferencia de puntos y fechas de partidos.  
Para esta libreta se utiliza como ejemplo la librería `nba_api`, que permite consultar estadísticas históricas de la NBA.

### Datos no estructurados
Son noticias deportivas, titulares y descripciones que pueden contener información contextual sobre lesiones, rendimiento, sanciones, estado anímico, alineaciones o eventos relevantes.  
Para esta libreta se prepara la conexión con **NewsAPI**.

### Nota importante
Si no se cuenta con una API Key activa de NewsAPI, la libreta genera automáticamente un dataset de respaldo para que el notebook pueda ejecutarse de principio a fin. Esto permite cumplir con el requisito de ejecución secuencial, pero en una entrega final real se recomienda sustituir la API Key por una válida.


## 🔹 2. Obtención de datos estructurados: partidos NBA

A continuación se obtiene información histórica de partidos de NBA usando `nba_api`.

Estos datos representan la parte estructurada del proyecto y pueden utilizarse para construir variables como:

- Fecha del partido.
- Puntos anotados.
- Diferencia de puntos.
- Resultado aproximado.
- Tendencias históricas.

Este bloque ayuda a representar la información de marcadores que el proyecto utilizaría como base cuantitativa.


In [ ]:
try:
    from nba_api.stats.endpoints import leaguegamefinder

    games_raw = leaguegamefinder.LeagueGameFinder().get_data_frames()[0]

    columns_to_keep = [
        "GAME_DATE", "TEAM_ID", "TEAM_ABBREVIATION", "TEAM_NAME",
        "MATCHUP", "WL", "PTS", "PLUS_MINUS"
    ]

    available_columns = [col for col in columns_to_keep if col in games_raw.columns]
    games = games_raw[available_columns].copy()

    games["GAME_DATE"] = pd.to_datetime(games["GAME_DATE"], errors="coerce")

    print("Datos NBA cargados correctamente.")
    display(games.head())

except Exception as e:
    print("No fue posible consultar nba_api. Se generará dataset estructurado de respaldo.")
    print("Motivo:", e)

    games = pd.DataFrame({
        "GAME_DATE": pd.date_range("2024-01-01", periods=120, freq="D"),
        "TEAM_ABBREVIATION": np.random.choice(["LAL", "BOS", "DEN", "MIA", "GSW", "NYK"], 120),
        "MATCHUP": np.random.choice(["LAL vs BOS", "DEN vs MIA", "GSW vs NYK"], 120),
        "WL": np.random.choice(["W", "L"], 120),
        "PTS": np.random.normal(112, 12, 120).round().astype(int),
        "PLUS_MINUS": np.random.normal(0, 14, 120).round().astype(int)
    })

    display(games.head())


## 🔹 3. Obtención de datos no estructurados: noticias deportivas

En esta sección se consultan noticias deportivas mediante **NewsAPI**.

Las noticias son relevantes para el proyecto porque pueden aportar información contextual que no aparece directamente en los marcadores, por ejemplo:

- Lesiones.
- Cambios de alineación.
- Malas rachas (por conflictos internos dentro del equipo).
- Estado emocional del equipo.
- Noticias de jugadores clave.
- Eventos externos que puedan impactar el desempeño.

Para ejecutar este bloque con datos reales, se debe reemplazar `"TU_API_KEY"` por una API Key válida de NewsAPI.

Si no se proporciona una API Key, se genera un dataset de respaldo con noticias de ejemplo para que la libreta siga funcionando de manera secuencial.


In [ ]:
API_KEY = "TU_API_KEY"

def load_sample_news():
    sample_data = [
        {
            "title": "Lakers star player questionable for tonight after ankle injury",
            "description": "The team will evaluate the player before the game, which could impact the betting market.",
            "publishedAt": "2026-05-01T10:00:00Z",
            "source": "Sample Sports News"
        },
        {
            "title": "Celtics continue winning streak with strong defensive performance",
            "description": "Analysts highlight improved defense and bench depth as key factors.",
            "publishedAt": "2026-05-01T13:30:00Z",
            "source": "Sample Basketball Daily"
        },
        {
            "title": "Denver faces travel fatigue before away game",
            "description": "The team arrives after consecutive road games and limited rest.",
            "publishedAt": "2026-05-02T09:15:00Z",
            "source": "Sample Sports Wire"
        },
        {
            "title": "Miami coach confirms lineup changes for next matchup",
            "description": "The adjustment may improve offensive spacing and transition defense.",
            "publishedAt": "2026-05-02T16:45:00Z",
            "source": "Sample Hoops Report"
        },
        {
            "title": "Warriors shooting performance raises expectations",
            "description": "Recent three-point efficiency has improved their projected scoring output.",
            "publishedAt": "2026-05-03T12:10:00Z",
            "source": "Sample Basketball Daily"
        },
        {
            "title": "Knicks deal with uncertainty after minor injury report",
            "description": "Two rotation players remain day-to-day according to local media.",
            "publishedAt": "2026-05-03T18:20:00Z",
            "source": "Sample Sports News"
        }
    ]
    return pd.DataFrame(sample_data)

try:
    if API_KEY == "TU_API_KEY":
        raise ValueError("API Key no configurada.")

    url = f"https://newsapi.org/v2/everything?q=nba basketball&language=en&pageSize=100&apiKey={API_KEY}"
    response = requests.get(url)
    data = response.json()

    if data.get("status") != "ok" or len(data.get("articles", [])) == 0:
        raise ValueError("NewsAPI no devolvió artículos válidos.")

    df_news = pd.DataFrame(data.get("articles", []))
    df_news = df_news[["title", "description", "publishedAt", "source"]].copy()
    df_news["source"] = df_news["source"].apply(lambda x: x.get("name") if isinstance(x, dict) else x)

    print("Noticias cargadas desde NewsAPI.")

except Exception as e:
    print("Se utilizará dataset de noticias de respaldo.")
    print("Motivo:", e)
    df_news = load_sample_news()

df_news["text"] = df_news["title"].fillna("") + " " + df_news["description"].fillna("")
df_news["publishedAt"] = pd.to_datetime(df_news["publishedAt"], errors="coerce")

display(df_news.head())


## 🔹 4. Revisión inicial de estructura del dataset

Antes de responder las preguntas exploratorias, se revisa la estructura general del dataset de noticias.

En este bloque veremos:

- Número de filas y columnas.
- Tipos de datos.
- Primeros registros.
- Columnas disponibles.

Esto permite validar que la información fue cargada correctamente antes de iniciar el análisis exploratorio.


In [ ]:
print("Dimensiones del dataset de noticias:", df_news.shape)
print("\nColumnas disponibles:")
print(df_news.columns.tolist())

print("\nTipos de datos:")
print(df_news.dtypes)

display(df_news.head())


## 🔹 5. ¿Hay valores faltantes en el conjunto de datos? ¿Se pueden identificar patrones de ausencia?

En esta sección se identifican valores faltantes dentro del dataset.

Esto es importante porque los modelos de Machine Learning y NLP pueden verse afectados si existen columnas incompletas, especialmente en campos de texto como título o descripción.

Se revisan:

- Cantidad de valores nulos por columna.
- Porcentaje de valores faltantes.
- Mapa visual de ausencia de datos.

Con esto se puede decidir si conviene eliminar, imputar o reconstruir campos faltantes.


In [ ]:
missing_count = df_news.isnull().sum()
missing_percent = (df_news.isnull().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent
})

display(missing_summary)

sns.heatmap(df_news.isnull(), cbar=False)
plt.title("Mapa de valores faltantes en noticias")
plt.show()


### Conclusión de valores faltantes

Si existen valores faltantes en columnas como `description`, estos pueden deberse a diferencias entre fuentes de noticias.  
Para NLP, una estrategia razonable es unir `title` y `description`, usando texto vacío cuando alguna descripción no esté disponible.  

Esto ya se aplica en la columna `text`, donde se combinan ambos campos para reducir pérdida de información.


## 🔹 6. ¿Cuáles son las estadísticas resumidas del conjunto de datos?

En esta sección se calculan estadísticas descriptivas.

Para variables categóricas se observan elementos como:

- Conteo.
- Valores únicos.
- Valor más frecuente.
- Frecuencia del valor más común.

Para variables numéricas, se revisan métricas como:

- Media.
- Desviación estándar.
- Mínimo.
- Máximo.
- Cuartiles.

Esto permite entender el comportamiento general de los datos.


In [ ]:
df_news["text_length"] = df_news["text"].astype(str).apply(len)

print("Estadísticas generales del dataset de noticias:")
display(df_news.describe(include="all"))

print("\nEstadísticas de longitud del texto:")
display(df_news["text_length"].describe())


## 🔹 7. ¿Hay valores atípicos en el conjunto de datos?

En esta sección se revisan posibles valores atípicos utilizando la variable `text_length`.

En NLP, textos demasiado cortos pueden aportar poca información, mientras que textos demasiado largos pueden introducir ruido o sesgo.

Se utiliza:

- Boxplot.
- Método IQR.
- Conteo de outliers.

Esto ayuda a decidir si se requiere limpieza adicional.


In [ ]:
sns.boxplot(x=df_news["text_length"])
plt.title("Valores atípicos en longitud de texto")
plt.xlabel("Longitud del texto")
plt.show()

Q1 = df_news["text_length"].quantile(0.25)
Q3 = df_news["text_length"].quantile(0.75)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = df_news[
    (df_news["text_length"] < lower_limit) |
    (df_news["text_length"] > upper_limit)
]

print("Límite inferior:", lower_limit)
print("Límite superior:", upper_limit)
print("Cantidad de outliers:", len(outliers))

display(outliers[["title", "description", "text_length"]].head())


### Conclusión de valores atípicos

Los outliers en longitud de texto deben revisarse antes del modelado.  
No necesariamente deben eliminarse, ya que una noticia larga podría contener información relevante, pero sí conviene evaluar si introduce ruido en el análisis.


## 🔹 8. ¿Cuál es la cardinalidad de las variables categóricas?

La cardinalidad indica cuántos valores únicos tiene una variable categórica.

En este proyecto, una variable categórica importante es `source`, ya que representa la fuente de la noticia.

Analizar la cardinalidad permite detectar:

- Si hay demasiadas fuentes diferentes.
- Si una sola fuente domina el dataset.
- Si puede existir sesgo de cobertura informativa.


In [ ]:
categorical_cols = df_news.select_dtypes(include=["object"]).columns.tolist()

cardinality = pd.DataFrame({
    "column": categorical_cols,
    "unique_values": [df_news[col].nunique() for col in categorical_cols]
})

display(cardinality)

print("Distribución de fuentes:")
display(df_news["source"].value_counts())

sns.countplot(y="source", data=df_news, order=df_news["source"].value_counts().index)
plt.title("Distribución de noticias por fuente")
plt.xlabel("Cantidad de noticias")
plt.ylabel("Fuente")
plt.show()


### Conclusión de cardinalidad

La fuente de noticias puede impactar el análisis debido a sesgo editorial o duplicidad de información.  
Si una fuente domina demasiado, podría afectar la generalización del modelo.


## 🔹 9. ¿Existen distribuciones sesgadas? ¿Necesitamos aplicar alguna transformación no lineal?

En esta sección se revisa la distribución de `text_length`.

El sesgo o `skewness` permite identificar si los datos están concentrados hacia un extremo.  
Si el sesgo es alto, puede considerarse una transformación no lineal, por ejemplo:

- Logaritmo natural con `log1p`.
- Raíz cuadrada.
- Winsorización.

En este análisis se compara la distribución original con una transformación logarítmica.


In [ ]:
sns.histplot(df_news["text_length"], bins=20, kde=True)
plt.title("Distribución original de longitud de texto")
plt.xlabel("Longitud del texto")
plt.show()

skew_original = df_news["text_length"].skew()
print("Skewness original:", skew_original)

df_news["log_text_length"] = np.log1p(df_news["text_length"])

sns.histplot(df_news["log_text_length"], bins=20, kde=True)
plt.title("Distribución transformada con log1p")
plt.xlabel("Log(1 + longitud del texto)")
plt.show()

skew_log = df_news["log_text_length"].skew()
print("Skewness después de log1p:", skew_log)


### Conclusión de sesgo

Si el sesgo original es alto y disminuye después de aplicar `log1p`, la transformación puede ser útil para algunos modelos.  
Sin embargo, para modelos basados en árboles como Random Forest, las transformaciones no siempre son obligatorias, aunque pueden ayudar en modelos lineales.


## 🔹 10. ¿Se identifican tendencias temporales?

Dado que el dataset incluye la columna `publishedAt`, se puede analizar la frecuencia de noticias a lo largo del tiempo.

Esto es relevante porque el volumen de noticias puede aumentar antes de partidos importantes, lesiones, playoffs o eventos deportivos relevantes.

Se revisa:

- Cantidad de noticias por fecha.
- Tendencia visual en el tiempo.


In [ ]:
df_news["published_date"] = df_news["publishedAt"].dt.date

news_by_date = df_news.groupby("published_date").size().reset_index(name="news_count")

display(news_by_date)

plt.plot(news_by_date["published_date"], news_by_date["news_count"], marker="o")
plt.title("Tendencia temporal de noticias")
plt.xlabel("Fecha")
plt.ylabel("Cantidad de noticias")
plt.xticks(rotation=45)
plt.show()


### Conclusión temporal

Las tendencias temporales ayudan a detectar momentos de mayor actividad informativa.  
En una versión más avanzada, estas fechas podrían cruzarse con partidos específicos para evaluar si el volumen o tono de noticias impacta la probabilidad de victoria.


## 🔹 11. Preparación de variables para correlación y análisis bivariado

Para poder analizar correlaciones y relaciones entre variables, se crean variables numéricas adicionales.

En esta versión exploratoria se crea una variable `sentiment_score` simulada.  
En una implementación productiva, esta variable debería obtenerse mediante servicios reales de NLP, como:

- Google Cloud Natural Language.
- AWS Comprehend.
- Azure Text Analytics.

También se crea una variable objetivo `target`, que representa una clase binaria simulada.  
En el proyecto final, esta variable debe sustituirse por el resultado real del partido, por ejemplo:

- 1 = victoria.
- 0 = derrota.

Esta simulación permite validar la estructura del pipeline mientras se integran fuentes reales completas.


In [ ]:
df_news["sentiment_score"] = np.random.choice([-1, 0, 1], size=len(df_news), p=[0.25, 0.50, 0.25])

# Variable objetivo simulada para validar el pipeline.
# Sustituir por resultados reales cuando se haga el cruce con partidos.
df_news["target"] = np.random.choice([0, 1], size=len(df_news), p=[0.48, 0.52])

display(df_news[["title", "text_length", "log_text_length", "sentiment_score", "target"]].head())


## 🔹 12. ¿Hay correlación entre las variables dependientes e independientes?

En esta sección se analiza la correlación entre variables numéricas:

- `text_length`
- `log_text_length`
- `sentiment_score`
- `target`

Esto permite identificar si existe una relación lineal inicial entre variables independientes y la variable objetivo.

Aunque una baja correlación no descarta utilidad predictiva, ayuda a entender relaciones simples entre variables.


In [ ]:
numeric_features = ["text_length", "log_text_length", "sentiment_score", "target"]

corr_matrix = df_news[numeric_features].corr()

display(corr_matrix)

sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de correlación")
plt.show()


### Conclusión de correlación

La correlación permite observar relaciones lineales preliminares.  
En problemas deportivos, muchas relaciones pueden ser no lineales, por lo que modelos como Random Forest pueden capturar patrones que una matriz de correlación no muestra directamente.


## 🔹 13. ¿Cómo se distribuyen los datos en función de diferentes categorías? Análisis bivariado

En esta sección se revisa cómo cambia la longitud del texto de acuerdo con el sentimiento simulado.

Este análisis ayuda a responder preguntas como:

- ¿Las noticias negativas tienden a ser más largas?
- ¿Las noticias positivas tienen una distribución diferente?
- ¿Existe alguna diferencia visible entre categorías?

El análisis bivariado es útil para identificar patrones entre una variable categórica y una variable numérica.


In [ ]:
sns.boxplot(x="sentiment_score", y="text_length", data=df_news)
plt.title("Longitud de texto por sentimiento")
plt.xlabel("Sentimiento simulado (-1 negativo, 0 neutral, 1 positivo)")
plt.ylabel("Longitud del texto")
plt.show()

bivariate_summary = df_news.groupby("sentiment_score")["text_length"].describe()
display(bivariate_summary)


### Conclusión bivariada

El análisis bivariado permite identificar diferencias entre grupos.  
En una versión final con sentimiento real, este análisis podría mostrar si las noticias negativas o positivas se relacionan con cambios en el desempeño o expectativa de resultado.


## 🔹 14. ¿Se deberían normalizar las imágenes para visualizarlas mejor?

Este proyecto no utiliza imágenes como fuente principal de información.

Por lo tanto:

- No aplica normalización de imágenes.
- No se requiere procesamiento visual.
- No se utilizan redes convolucionales ni datasets de imágenes.

Sin embargo, si en el futuro se incorporaran imágenes, como fotografías de jugadores, canchas o mapas de calor, sí sería recomendable normalizarlas para mejorar visualización y entrenamiento de modelos.


## 🔹 15. ¿Hay desequilibrio en las clases de la variable objetivo?

En esta sección se revisa el balance de la variable `target`.

El desbalance de clases puede afectar el desempeño del modelo, ya que un modelo podría aprender a predecir principalmente la clase mayoritaria.

En el proyecto final, esta variable deberá construirse con resultados reales de partidos.  
Por ahora se utiliza una variable simulada para demostrar el análisis.


In [ ]:
target_distribution = df_news["target"].value_counts().sort_index()
target_percentage = df_news["target"].value_counts(normalize=True).sort_index() * 100

balance_summary = pd.DataFrame({
    "count": target_distribution,
    "percentage": target_percentage.round(2)
})

display(balance_summary)

sns.countplot(x="target", data=df_news)
plt.title("Balance de clases de la variable objetivo")
plt.xlabel("Target")
plt.ylabel("Cantidad")
plt.show()


### Conclusión de balance de clases

Si una clase domina de forma importante, se deben considerar técnicas como:

- Recolección de más datos.
- Submuestreo.
- Sobremuestreo.
- Ajuste de pesos por clase.
- Métricas adicionales como F1-score, recall o AUC.

Esto es especialmente relevante en predicción deportiva, donde el objetivo no es solo acertar, sino estimar probabilidades útiles para la toma de decisiones.


## 🔹 16. NLP: representación del texto con TF-IDF

En esta sección se convierte el texto en variables numéricas usando TF-IDF.

TF-IDF permite identificar palabras relevantes dentro del conjunto de noticias, dando más peso a términos importantes y menos peso a palabras demasiado frecuentes.

Esto es útil para representar noticias deportivas como entrada para modelos de Machine Learning.


In [ ]:
tfidf = TfidfVectorizer(
    max_features=20,
    stop_words="english"
)

X_tfidf = tfidf.fit_transform(df_news["text"].fillna(""))

tfidf_terms = pd.DataFrame({
    "term": tfidf.get_feature_names_out()
})

display(tfidf_terms)


## 🔹 17. Modelo base para validar el pipeline

Aunque el entregable principal es el análisis exploratorio, se incluye un modelo base para validar que los datos pueden alimentar un flujo de Machine Learning.

Se utiliza Random Forest porque:

- Puede manejar relaciones no lineales.
- Es robusto para variables numéricas.
- Sirve como baseline inicial.

Importante: en esta versión, `target` y `sentiment_score` son variables simuladas.  
Por lo tanto, los resultados del modelo no deben interpretarse como desempeño real del proyecto, sino como validación del pipeline.


In [ ]:
df_model = df_news[["text_length", "log_text_length", "sentiment_score", "target"]].dropna()

X = df_model[["text_length", "log_text_length", "sentiment_score"]]
y = df_model["target"]

if len(df_model) >= 5 and y.nunique() > 1:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
    )

    model = RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print("\nMatriz de confusión:")
    print(confusion_matrix(y_test, y_pred))
    print("\nReporte de clasificación:")
    print(classification_report(y_test, y_pred))
else:
    print("No hay suficientes datos o clases para entrenar el modelo.")


## 🔹 18. ¿Qué falta para usar datos reales en el modelo final?

Para convertir este notebook exploratorio en un modelo predictivo real, se deben sustituir las variables simuladas por datos reales:

### Sentimiento real
Se puede obtener con:
- Google Cloud Natural Language.
- AWS Comprehend.
- Azure Text Analytics.

### Variable objetivo real
Se debe construir cruzando noticias con partidos reales.  
Por ejemplo:

- Tomar fecha de noticia.
- Asociarla a un partido cercano.
- Obtener resultado real del partido desde NBA API o API-FOOTBALL.
- Crear target:
  - `1 = victoria`
  - `0 = derrota`

### Odds o cuotas
Para análisis de apuestas, conviene agregar:
- Cuotas iniciales.
- Cuotas de cierre.
- Probabilidad implícita.
- Diferencia entre probabilidad del modelo y probabilidad del mercado.

### Modelos futuros
Además de Random Forest, el proyecto puede explorar:
- Poisson para goles/canastas esperadas.
- ELO para ranking dinámico.
- Regresión logística.
- XGBoost.
- Redes neuronales.


## 🔹 19. Conclusión general

El análisis exploratorio permitió revisar la estructura, calidad y comportamiento inicial de los datos del proyecto **Sports Bet Engine**.

Se cubrieron los puntos principales solicitados:

- Identificación de valores faltantes.
- Estadísticas resumidas.
- Detección de valores atípicos.
- Cardinalidad de variables categóricas.
- Distribuciones sesgadas y posible transformación no lineal.
- Tendencias temporales.
- Correlación entre variables.
- Análisis bivariado.
- Justificación sobre normalización de imágenes.
- Revisión del balance de clases.

La integración de datos estructurados de marcadores y datos no estructurados de noticias permite enriquecer el análisis predictivo, ya que incorpora tanto desempeño histórico como contexto dinámico.
